[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/adairgranados/adair-granados?quickstart=1)


<!-- gen:values.yml#weeks.08.notebook-title|Demonstration -->
# Week 8 Demonstration: DataViz II: Storytelling, decluttering, and accessibility
<!-- /gen -->

*This week, we take one chart from an [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate)'s exploratory draft to a figure for a board pack, checking each step as we go.*

## The job

**Hartwell's Garden Centres** has six branches across Surrey and Berkshire. It has asked [Runnymede](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-runnymede-analytics) which branch drives growth, before it decides where next year's refurbishment budget goes. The analysis is done. This notebook turns the exploratory chart into the figure that goes in front of Hartwell's board.

Run every cell. Where you see an **[AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens)**, try its prompt.

The cell below builds Hartwell's three years of monthly revenue by branch. Run it.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(5813)  # module seed: Do not change

# Hartwell's Garden Centres: Monthly revenue (£) per branch, Jun 2023 to May 2026.
# I built the data so that Egham grows strongly, and faster once its glasshouse
# café opens in March 2025. The other five branches stay flat or drift down.
months = pd.date_range("2023-06-01", periods=36, freq="MS")
t = np.arange(36)
season = 1 + 0.05 * np.sin(2 * np.pi * (months.month - 4) / 12)  # spring peak

def branch(start_k, end_k, cafe_bump_k=0.0, open_month=21):
    trend = np.linspace(start_k, end_k, 36)
    ramp = np.clip((t - open_month) / 4, 0, 1) * cafe_bump_k
    pounds = (trend + ramp) * season * 1000 + rng.normal(0, 1100, 36)
    return np.round(pounds, -2)  # to the nearest £100

rev = pd.DataFrame({
    "Egham":     branch(24, 44, cafe_bump_k=10),
    "Staines":   branch(42, 44),
    "Windsor":   branch(44, 36),
    "Guildford": branch(27, 31),
    "Woking":    branch(28, 26),
    "Camberley": branch(21, 20),
}, index=months)

CAFE_OPENS = pd.Timestamp("2025-03-01")
print(f"Created rev: {rev.shape[0]} months x {rev.shape[1]} branches (monthly revenue, £), in-memory, seed 5813")

## 1. The exploratory draft

The chart below is the Associate's answer to a one-line prompt. At the exploration stage that is enough: You want to see the data.

> **Prompt that produced it:** *"plot monthly revenue by branch for Hartwell's"*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for b in rev.columns:
    ax.plot(rev.index, rev[b], linewidth=1.8, label=b)
ax.set_facecolor("#eaeaf2")
ax.grid(True, color="white", linewidth=1.0)
ax.legend(loc="upper left", ncols=2)
ax.set_ylabel("value")
ax.set_title("Monthly Revenue by Branch")
plt.show()

You can see six lines, and with effort you can see that one of them climbs. As a board figure, the draft fails most letters of [TRACE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-trace). It has six competing colours, a legend sitting on the data, a tinted panel, and a full grid. Its y-axis is labelled `value` and shows raw pounds. Its title names the variables.

> 🤖 **AI Lens:** Paste the code cell above into the prompt below, in place of the placeholder, and send it to your Associate. Compare its critique with the paragraph above. Check anything it passes yourself.

```text
<paste the cell's code here>

Run this figure through TRACE (Task, Reduce, Accessible, Context, Explain). Where does it fail?
```

**A good answer** fails the figure on most letters and matches the paragraph above.

## 2. The declutter sequence

The [design guide, section 3](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), sets out the declutter sequence. The next cell wraps the chart in a function, `branch_chart()`, with one setting: `upto=n` applies the sequence up to step *n*. Each block in the function is one step, and each comment is the short instruction you give your Associate.

Step 4 writes each branch name beside its line with [`ax.annotate`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.annotate.html). In step 5, `plt.MultipleLocator(10)`, a matplotlib [tick locator](https://matplotlib.org/stable/api/ticker_api.html), puts a y-tick every £10k.

In [ ]:
STEP_NAMES = {
    0: "The Associate's default",
    1: "Spines",
    2: "Gridlines",
    3: "Background",
    4: "Legend → direct labels",
    5: "Rounding and axis density",
    6: "Emphasis",
}

def branch_chart(ax, upto=0, accent="#d62728"):
    """Hartwell's revenue chart, decluttered up to step `upto` (0-6)."""
    data = rev / 1000 if upto >= 5 else rev   # step 5 rescales £ to £k

    for b in data.columns:
        if upto >= 6:  # Step 6: "grey for context; one strong colour for the story"
            main = b == "Egham"
            ax.plot(data.index, data[b],
                    color=accent if main else "#c9c9c9",
                    linewidth=2.6 if main else 1.2,
                    zorder=3 if main else 2)
        else:
            ax.plot(data.index, data[b], linewidth=1.8, label=b)

    if upto >= 1:  # Step 1: "remove the box; keep only the bottom and left spines"
        ax.spines[["top", "right"]].set_visible(False)

    if upto >= 2:  # Step 2: "delete the grid; add back a few faint horizontal lines"
        ax.grid(axis="y", color="#b3b3b3", linewidth=0.8)
        ax.set_axisbelow(True)
    else:
        ax.grid(True, color="white", linewidth=1.0)

    if upto < 3:   # Step 3: "white background; lose the tinted panel"
        ax.set_facecolor("#eaeaf2")

    if upto >= 4:  # Step 4: "delete the legend box; name each line at its right-hand end"
        for b in data.columns:
            main = upto >= 6 and b == "Egham"
            ax.annotate(b, (data.index[-1], data[b].iloc[-3:].mean()),
                        xytext=(8, 0), textcoords="offset points",
                        va="center", fontsize=9,
                        color=(accent if main else "#767676") if upto >= 6 else "black",
                        fontweight="bold" if main else "normal")
        ax.margins(x=0.02)
    else:
        ax.legend(loc="upper left", ncols=2)

    if upto >= 5:  # Step 5: "round the numbers; say what the axis measures, in units"
        ax.set_ylabel("Monthly revenue (£k)")
        ax.yaxis.set_major_locator(plt.MultipleLocator(10))
    else:
        ax.set_ylabel("value")

    ax.set_title(f"After step {upto}: {STEP_NAMES[upto]}" if upto else
                 f"Before: {STEP_NAMES[0]}", loc="left")

def pair(step):
    """Draw the chart immediately before and after declutter step `step`."""
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
    branch_chart(axes[0], upto=step - 1)
    branch_chart(axes[1], upto=step)
    fig.autofmt_xdate(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

print("Defined branch_chart() and pair()")

*(Remember: You're not required to write the above yourself. This example uses many advanced Python concepts. However, you are expected to ask your Associate to explain it to you. What you will learn is a really elegant way to create charts programmatically. It's worth putting effort into this, you will be able to reuse it many times over!)* 

### Step 1: Spines

> *"Remove the box around the plot; keep only the bottom and left spines."*

The top and right spines only draw a frame.

In [ ]:
pair(1)

### Step 2: Gridlines

> *"Delete the grid, then add back only a few faint horizontal lines."*

Vertical gridlines rarely help a monthly line chart, and heavy ones draw the eye away from the data.

In [ ]:
pair(2)

### Step 3: Background

> *"White background: Lose the tinted panel."*

The tinted panel is a style default. On a slide or a printed page it looks like a grey box around your figure.

In [ ]:
pair(3)

### Step 4: Legend → direct labels

> *"Delete the legend box and name each line at its right-hand end."*

With a legend, the reader has to look from each line to the key and back, six times. Direct labels put each name next to its line. This week's [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) stage shows what is lost when a legend goes and no labels replace it.

In [ ]:
pair(4)

### Step 5: Rounding and axis density

> *"Plot in £k; label the axis with what it measures and its units; round the ticks."*

With `40000`, the reader has to count the zeros. "40" on an axis labelled "Monthly revenue (£k)" is quicker to read.

In [ ]:
pair(5)

### Step 6: Emphasis last

> *"Push the five context branches into grey; pull Egham into one strong colour."*

Hartwell's wants to know which branch drives growth. The answer is Egham. The figure makes Egham stand out and keeps the other five branches as context.

In [ ]:
pair(6)

Compare `pair(6)`'s right-hand panel with the Associate's original.

> 🤖 **AI Lens:** Run the sequence again, this time by directing your Associate. Give it the original draft and the instructions above one at a time, and check each result before the next. The first prompt below opens that session; follow it with steps 2 to 6 in turn. Then start a fresh chat and give every step at once with the second prompt. Compare the two finished charts, then the two chats. The step-by-step session shows you when the Associate deletes something you did not ask it to delete. In the one-shot session, you are less likely to notice.

```text
<paste the original draft chart code here>

I am going to give you six changes to this chart, one at a time. Apply only the change I name, show the full revised code, and change nothing else. Step 1: Remove the box around the plot; keep only the bottom and left spines.
```

```text
<paste the original draft chart code here>

Apply all six of these changes to this chart and show the full revised code:
1. Remove the box around the plot; keep only the bottom and left spines.
2. Delete the grid, then add back only a few faint horizontal lines.
3. White background: Lose the tinted panel.
4. Delete the legend box and name each line at its right-hand end.
5. Plot in £k; label the axis with what it measures and its units; round the ticks.
6. Push the five context branches into grey; pull Egham into one strong colour.
```

**A good answer** looks like `pair(6)`'s right-hand panel, whichever route you take. If the step-by-step chat shows you catching a change you did not ask for, paste it into your [Prompt Diary](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-prompt-diary).

## 3. The accessibility pass

The figure is now decluttered. Next, make it work for everyone in the room. The step-6 accent is matplotlib's default red, a poor choice here for these reasons:

- It is one of the colours most affected by common colour-vision deficiencies
- It fades in greyscale print
- To a finance audience, red means loss

The pass follows the [design guide, section 4](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md):

1. **Palette:** Swap the accent to `#0072B2`, the blue from the Okabe-Ito colour-blind-safe palette. Context stays grey
2. **Second channel:** The direct labels, the heavier line, and Egham's position show which line matters without relying on colour
3. **Contrast:** Label text in `#767676` on white sits at the WCAG AA threshold (about 4.5:1). A lighter grey fails it
4. **Alt text:** One sentence, written below

The cell below draws the red accent and the blue accent side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
branch_chart(axes[0], upto=6)                      # matplotlib's default red
branch_chart(axes[1], upto=6, accent="#0072B2")    # Okabe-Ito blue
axes[0].set_title("Before: Default red accent", loc="left")
axes[1].set_title("After: Okabe-Ito blue accent", loc="left")
fig.autofmt_xdate(rotation=30, ha="right")
plt.tight_layout()
plt.show()

The alt text is a variable in the code, so it goes with the figure into your report.

In [ ]:
# Alt text: Chart type + variables + takeaway, in one sentence
alt_text = (
    "Line chart of monthly revenue (£k) for Hartwell's six garden-centre branches, "
    "June 2023 to May 2026, showing Egham roughly doubling (and accelerating after "
    "its café opened in March 2025) while the other five branches stay flat or "
    "drift slightly down."
)
print(alt_text)

> 🤖 **AI Lens:** Ask your Associate to draft the alt text for this figure with the prompt below. Check its draft against the formula *(type + variables + takeaway)*. Your Associate's drafts often describe the chart's parts ("a line chart with six coloured lines") and leave out the finding. Fix the draft, and note in one line what you changed. Put that line in your [design rationale](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-design-rationale).

```text
<paste the figure's code here>

Draft one-sentence alt text for this figure.
```

**A good answer** names the chart type, the variables and their units, and the takeaway, as the alt text above does. The takeaway is that Egham roughly doubles while the other five branches stay flat.

## 4. Annotation and the takeaway title: The finished figure

Next, mark the café opening, the event behind the acceleration. Then write a takeaway title that states the finding. The cell below writes out the final figure's code in full, without `branch_chart()`. You can copy the pattern into your own work. It marks the opening with [`ax.axvline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axvline.html) and a labelled arrow, then adds the takeaway title, a subtitle, and a source line. This figure goes in the board pack, with the alt text as its caption.

In [ ]:
data = rev / 1000
ACCENT, CONTEXT, TEXT_GREY = "#0072B2", "#c9c9c9", "#767676"

fig, ax = plt.subplots(figsize=(9.5, 5))
for b in data.columns:
    main = b == "Egham"
    ax.plot(data.index, data[b],
            color=ACCENT if main else CONTEXT,
            linewidth=2.6 if main else 1.2, zorder=3 if main else 2)
    ax.annotate(b, (data.index[-1], data[b].iloc[-3:].mean()),
                xytext=(8, 0), textcoords="offset points", va="center",
                fontsize=9, color=ACCENT if main else TEXT_GREY,
                fontweight="bold" if main else "normal")

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.margins(x=0.02)
ax.set_ylabel("Monthly revenue (£k)")
ax.yaxis.set_major_locator(plt.MultipleLocator(10))

# Annotation: Mark the event that explains the acceleration
ax.axvline(CAFE_OPENS, color=TEXT_GREY, linewidth=0.8, linestyle=":")
ax.annotate("Glasshouse café opens\n(Mar 2025)",
            xy=(CAFE_OPENS, data.loc[CAFE_OPENS, "Egham"]),
            xytext=(-110, 55), textcoords="offset points",
            fontsize=9, color="#444444",
            arrowprops=dict(arrowstyle="->", color=TEXT_GREY))

# Takeaway title + descriptive subtitle
ax.set_title("Egham now drives all of Hartwell's growth",
             loc="left", fontsize=13, fontweight="bold", pad=26)
ax.text(0, 1.03, "Monthly revenue by branch (£k), June 2023–May 2026",
        transform=ax.transAxes, fontsize=9.5, color="#555555")

# Source line: What makes the figure checkable
fig.text(0.07, -0.02,
         "Source: Hartwell's till-system exports, Jun 2023–May 2026; revenue net of VAT.",
         fontsize=8, color=TEXT_GREY)
plt.show()

The exploratory draft and this figure use the same data and the same library. The difference is a series of small <!-- plain-language: everyday decision -->decisions that you can name and explain. The design rationale ([design guide, section 6](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md)) asks you to write them down.

## 5. Aside: Interactive charts with hvPlot

Everything above is a static figure, the right form for a report or a board slide. An **interactive** chart lets the reader hover for exact values, zoom into a quarter, and hide or show lines. The quickest route from a `DataFrame` is [hvPlot](https://hvplot.holoviz.org/), which offers the same `.plot`-style call, rendered as a live Bokeh chart. The cell below draws one.

In [ ]:
import hvplot.pandas  # noqa: F401  (registers .hvplot on DataFrames)

(rev / 1000).hvplot.line(
    height=350, width=720,
    ylabel="Monthly revenue (£k)", xlabel="",
    group_label="Branch",
    title="Hartwell's monthly revenue by branch: Hover, zoom, toggle the legend",
)

The first run takes a few seconds while the plotting engine loads. Use the toolbar to zoom, and click legend entries to hide or show lines.

**When interactivity helps:** An interactive chart helps while you explore the data. With it, you can answer questions that the static figure does not cover: "What was Windsor in November 2024?" takes one hover. It also helps when you sit with a stakeholder who keeps asking "what about…?"

**When to avoid interactivity:** Your individual report, a PDF board pack, and a printed slide are all static. In each of these, an interactive chart shows only its default view. Hover, zoom, and toggles stop working, and you lose control of what the reader sees first. Explore with interactive charts. Publish static figures, unless you are asked to build a dashboard. Neither of this module's assessments asks for one.

> 🤖 **AI Lens:** Ask your Associate the first question below, with the hvPlot cell pasted in place of the placeholder. Then ask the second.

```text
<paste the hvPlot cell's code here>

The CFO gets this figure in a printed board pack. What breaks if I ship the hvPlot version?
```

```text
Beyond hover, zoom, and widgets failing on paper: What do I lose control of if the hvPlot version goes in a printed board pack?
```

**A good answer** to the first lists hover, zoom, and widgets, which stop working on paper. A good answer to the second: You lose control of the reader's first impression, the framing, and the emphasis.

## Where you stand

In **[Quest 08](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest): The Boardroom Story**, you take four exploratory charts through the same steps to one slide for a CFO. The quest contains its own declutter checklist.